# Clase 5 — Práctica Individual
## Tu turno: auditar el archivo
### Maestría en Fintech · ITBA · 2026

---

El mismo archivo de la práctica guiada: `transacciones_sucias.csv`. **Preguntas nuevas.**

Además tenés `clientes` y `tarjetas` (para validar contra otras tablas, como en la Clase 4) y la `referencia`:
la tabla conciliada de las clases 3 y 4, para medir cuánto te acercaste.

> **La regla del día:** antes de borrar algo, preguntate **qué es** y **qué pregunta querés contestar**. Y anotá lo que decidiste.

---
## Antes de empezar: guardá tu copia

Abriste esta notebook desde el link del curso, así que nadie pisa el trabajo de nadie. Pero **lo que escribas acá no se guarda**.

Para quedarte con tu trabajo (y poder terminarla en casa): **`Archivo`** → **`Guardar una copia en Drive`**,
con el nombre **`Clase5_Individual_TuNombre`**.

---
## Cómo se trabaja acá

**Una pregunta por vez.** Escribí, ejecutá, mirá el resultado, y recién ahí seguí.

**Cada respuesta termina con una lectura de negocio**, en la celda `**Lectura:**` de abajo. Un número solo no es una respuesta.

**Gemini es tu copiloto, no tu piloto.** Pedile sintaxis y que te explique errores. **No le pidas que limpie el archivo**:
va a borrar lo que le parezca raro, y ya viste en la guiada adónde lleva eso.

**Un prompt útil para limpieza describe el problema y la decisión que ya tomaste:**

> *"Tengo un DataFrame `df` con una columna `monto_ars`. Quiero completar los nulos de `monto_ars` con el promedio de su `categoria`,
> y agregar una columna booleana `monto_imputado` que marque cuáles completé."*

Y siempre: **leé el código antes de ejecutarlo**. Fijate si borra filas que no le pediste.

---
## Setup

Ejecutá las dos celdas. La primera carga las tablas. La segunda repite **la limpieza de la guiada** y te deja una tabla `limpio`
(la vas a usar en los ejercicios 3, 7, 8 y 9). `sucio` queda intacto.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

DATOS = 'https://raw.githubusercontent.com/camilojaure/itba-pad/main/datasets/'
sucio      = pd.read_csv(DATOS + 'transacciones_sucias.csv')
clientes   = pd.read_csv(DATOS + 'clientes.csv')
tarjetas   = pd.read_csv(DATOS + 'tarjetas.csv')
referencia = pd.read_csv(DATOS + 'transacciones.csv')   # la tabla conciliada de las clases 3 y 4

print(f'sucio:      {len(sucio):>6,} filas')
print(f'clientes:   {len(clientes):>6,} filas')
print(f'tarjetas:   {len(tarjetas):>6,} filas')
print(f'referencia: {len(referencia):>6,} filas')

In [ ]:
# La limpieza de la práctica guiada, en una sola celda
limpio = sucio.drop_duplicates().copy()

limpio['categoria'] = limpio['categoria'].str.strip().str.capitalize()
limpio['tipo'] = (limpio['tipo'].str.strip().str.lower()
                  .replace({'credito': 'crédito', 'credit': 'crédito'}))

limpio = limpio[~(limpio['monto_ars'] <= 0)].copy()

veces = limpio['monto_ars'] / limpio.groupby('cliente_id')['monto_ars'].transform('median')
limpio.loc[veces > 20, 'monto_ars'] = np.nan

limpio['categoria'] = limpio['categoria'].fillna('Sin categoría')
limpio['monto_imputado'] = limpio['monto_ars'].isna()
limpio['monto_ars'] = limpio['monto_ars'].fillna(limpio.groupby('categoria')['monto_ars'].transform('mean'))

print(f"limpio: {len(limpio):,} filas · TPV ${limpio['monto_ars'].sum() / 1e6:,.0f} M "
      f"· {limpio['monto_imputado'].sum():,} montos imputados")

---
## Ejercicio 1 · Tu propio checklist

Sobre `sucio`, calculá:

- el **porcentaje de nulos** de cada columna
- cuántas filas tienen **al menos un problema**: algún nulo, o están duplicadas, o tienen monto menor o igual a cero

¿Qué porcentaje del archivo es eso?

*Pista: `sucio.isna().any(axis=1)` te dice, fila por fila, si tiene algún nulo. Combiná las tres condiciones con `|` (o).*

In [ ]:
# Tu código acá

**Lectura:** *Si la regla fuera "borro toda fila que tenga algún problema", ¿qué parte del negocio perdés? ¿Te animás a mandar un número así?*

> _(escribí tu respuesta acá)_

---
## Ejercicio 2 · ¿Los nulos están repartidos al azar?

Uní `sucio` con `clientes` para traer el segmento, y calculá el **porcentaje de montos nulos en cada segmento**.
Hacé lo mismo **por mes** (convertí la fecha primero).

*Pista: el promedio de `monto_ars.isna()` es la proporción de nulos, el mismo truco de la Clase 2 con una columna 0/1.*

In [ ]:
# Tu código acá

**Lectura:** *¿Los nulos se concentran en algún segmento o mes? ¿Por qué eso importa para decidir si eliminarlos o no? Pensá qué pasaría si todos los nulos fueran de clientes Premium.*

> _(escribí tu respuesta acá)_

---
## Ejercicio 3 · Débito y crédito, antes y después

¿Qué porcentaje del **monto** se paga con débito y qué porcentaje con crédito?
Calculalo dos veces: sobre `sucio` **sin tocar** y sobre `limpio`.

*Pista: `groupby('tipo')['monto_ars'].sum()` y dividí por el total.*

In [ ]:
# Tu código acá

**Lectura:** *¿Cuántas filas tiene la tabla "antes"? Si le mandaras esa tabla a un gerente comercial, ¿qué conclusión equivocada podría sacar?*

> _(escribí tu respuesta acá)_

---
## Ejercicio 4 · Los montos negativos

Mirá de cerca las transacciones con **monto negativo** en `sucio` (sin duplicados):
¿cuántas son, en qué categorías, de qué tipo de tarjeta, están aprobadas? ¿Cuánto suman?

*Pista: filtrá y usá `value_counts()` sobre cada columna que te interese.*

In [ ]:
# Tu código acá

**Lectura:** *¿Son devoluciones o errores? ¿Qué dato te falta para saberlo, y a qué área se lo pedirías? ¿Qué hubieras hecho si fueran 3.000 y no 14?*

> _(escribí tu respuesta acá)_

---
## Ejercicio 5 · El límite de la tarjeta como validación

Una compra con tarjeta de **crédito** no debería superar el **límite** de esa tarjeta. Usalo para validar los montos:

1. Quedate con las transacciones de `sucio` sin duplicados y con `tipo` normalizado a `crédito`
2. Uní con `tarjetas` para traer el `limite_credito` (la clave es `tarjeta_id`)
3. ¿Cuántas superan el límite? Para cada una, calculá **cuántas veces el límite** es el monto

*Pista: es un join de la Clase 4. Antes de unir, predecí cuántas filas tienen que salir, y después contalas.*

In [ ]:
# Tu código acá

**Lectura:** *¿Todas las que superan el límite son errores? ¿Qué diferencia hay entre pasarse un 10% y pasarse 20 veces? ¿Te sirve esta regla para las compras con débito?*

> _(escribí tu respuesta acá)_

---
## Ejercicio 6 · Imputar tiene un precio

Partiendo de `sucio` sin duplicados, sin montos no positivos y con la categoría normalizada, compará **tres formas** de tratar los montos nulos:

| Estrategia | Qué hace |
|---|---|
| Eliminar | Se descartan las filas sin monto |
| Mediana de la categoría | Se completan con la mediana del monto de su categoría |
| Promedio de la categoría | Se completan con el promedio del monto de su categoría |

Para cada una, calculá el **TPV** y el **desvío estándar** del monto. Comparalos con la `referencia`.
(Para que la comparación sea justa, antes sacá los **montos gigantes** como en la guiada.)

*Pista: `groupby('categoria')['monto_ars'].transform('median')` te da, para cada fila, la mediana de su categoría.*

In [ ]:
# Tu código acá

**Lectura:** *¿Qué estrategia se acerca más al TPV real? ¿Qué le pasa al desvío estándar cuando imputás? ¿Para qué pregunta de negocio preferirías eliminar?*

> _(escribí tu respuesta acá)_

---
## Ejercicio 7 · El año que no fue

Graficá el **TPV mensual** de 2024 en el mismo gráfico, dos líneas:

- `sucio` tal como viene (sin limpiar nada)
- `limpio`

Las filas sin fecha no entran en ninguna de las dos: no se pueden ubicar en un mes.

*Pista: `pd.to_datetime` y `.dt.month`, como en la Clase 3. Armá las dos series y juntalas en un DataFrame para graficar.*

In [ ]:
# Tu código acá

**Lectura:** *Con el archivo sucio, ¿cuál parece el mejor mes del año? ¿Y con el limpio? En la Clase 3 vimos que diciembre es el pico: ¿qué le hubieras contado al directorio con el archivo sin limpiar? ¿Cuánto TPV quedó sin mes?*

> _(escribí tu respuesta acá)_

---
## Ejercicio 8 · El ticket promedio por categoría

Armá una tabla con el **ticket promedio por categoría** en tres columnas: `sucio` (con la categoría normalizada, pero sin ninguna otra limpieza),
`limpio` (solo los montos **no imputados**) y `referencia`. Agregá el **error del sucio** en %.

*Pista: tres `groupby` y un `pd.DataFrame({...})` para juntarlos, como en el Paso 7 de la guiada.*

In [ ]:
# Tu código acá

**Lectura:** *¿Qué categoría está más distorsionada en el archivo sucio? ¿Cuántas filas explican esa distorsión? ¿Por qué en el limpio usamos solo los montos que no imputamos?*

> _(escribí tu respuesta acá)_

---
## Ejercicio 9 · Las tablas, ¿cierran entre sí?

Un chequeo que no se hace mirando una sola tabla. Sobre `limpio`, verificá:

1. ¿Todos los `cliente_id` de las transacciones existen en `clientes`?
2. ¿Todas las `tarjeta_id` existen en `tarjetas`?
3. ¿Cada tarjeta pertenece **al mismo cliente** en las dos tablas?
4. ¿El `tipo` de la transacción coincide con el `tipo_tarjeta` de la tarjeta (débito con débito, crédito con crédito)?

*Pista: para 1 y 2, `isin`. Para 3 y 4, un `merge` con `tarjetas` por `tarjeta_id` y comparar columnas.*

In [ ]:
# Tu código acá

**Lectura:** *¿Qué encontraste? Si todo cierra, ¿vale la pena haberlo chequeado? ¿En qué situaciones de tu trabajo esperarías que no cierre?*

> _(escribí tu respuesta acá)_

---
## Ejercicio 10 · Desafío: tu herramienta para el lunes

Escribí una función **`reporte_calidad(df)`** que reciba **cualquier** DataFrame y devuelva una tabla con **una fila por columna** y estos datos:

| Columna del reporte | Qué dice |
|---|---|
| `tipo` | El tipo de dato de la columna |
| `nulos` y `% nulos` | Cuántos valores faltan |
| `distintos` | Cuántos valores distintos tiene |
| `ceros o negativos` | Solo para columnas numéricas: cuántos valores son ≤ 0 |
| `max / p99` | Solo para columnas numéricas: cuántas veces el máximo supera al percentil 99 |

Además, que **imprima** cuántas filas duplicadas tiene el DataFrame.

Probala con `sucio`, con `limpio` y con `clientes`.

*Pistas: armá una lista de diccionarios (uno por columna) y al final `pd.DataFrame(lista)`, como la bitácora de la guiada.
`pd.api.types.is_numeric_dtype(df[col])` te dice si una columna es numérica. Pedile a Gemini la estructura, pero decidí vos qué medir.*

In [ ]:
def reporte_calidad(df):
    # Tu código acá
    pass

# Probala:
# reporte_calidad(sucio)

In [ ]:
# reporte_calidad(limpio)
# reporte_calidad(clientes)

**Tu mensaje para quien te pasó el archivo — tres bullets, no una tabla:**

1. **Qué encontraste:** _(los problemas del archivo, con números)_
2. **Qué decidiste:** _(qué hiciste con cada uno, y cuánto movió el TPV)_
3. **Qué pedís:** _(qué tendría que cambiar en el origen para que el próximo archivo llegue mejor)_

---
## Antes de cerrar — checklist

- [ ] Guardaste **tu copia** en Drive
- [ ] Todas las celdas corren de arriba a abajo sin errores (`Entorno de ejecución → Reiniciar y ejecutar todo`)
- [ ] Antes de borrar filas, **contaste cuántas** y te preguntaste qué eran
- [ ] Cada ejercicio tiene su **lectura de negocio** escrita, no solo el número
- [ ] `reporte_calidad` funciona con **cualquier** DataFrame, no solo con `sucio`

---

**Si algo no salió:** anotá dónde te trabaste y traelo a la próxima clase.

**Qué viene en la Clase 6:** el examen integrador y qué sigue después del curso. Se evalúa el criterio, no la sintaxis: qué harías con un nulo, qué join elegirías, cómo leerías un resultado.